# GEMS-Bench — Generator

**G**erman **M**ulti-hop **S**peech Benchmark. Builds the fully closed-world retrieval benchmark stage by stage.

Real DM products (public Search API) = only name/category skeleton. Brand, supplier, warehouse, region, buyer, all attributes and all answers are **invented & deterministic from seed** → parametrically unanswerable, deterministically scoreable (no LLM judge).

Data model: `README.md`. All knobs: `config.yaml`

## About this notebook

This notebook is the audit trail for how the dataset was built, and the mechanism for regenerating a variant with a different `seed` or size. The committed artifacts under `data/` are the canonical dataset — running this notebook is not required to use the benchmark.

Regeneration is **not bit-identical**: the LLM stages run at `llm.temperature` 0, but through a hosted endpoint, so exact reproduction across runs is not guaranteed.

`get_products.py` (Stage 0) is the only stage that touches the network — it fetches the product skeleton from the public DM Search API — and it cannot reproduce the original fetch exactly, because that catalog changes over time. Every stage from Stage 1 onward is deterministic from `seed`.

## Requirements to run

- An LLM endpoint reachable via the `llm.endpoint_var` / `llm.key_var` environment variables (`AZURE_ENDPOINT` / `AZURE_API_KEY` by default) for the name, corpus-weave, spoken-rewrite, and closed-book-gate stages.
- A local TTS model (`tts.model_id`) to synthesize audio.
- A local Whisper model (`stt.model`) for the ASR verification steps.

## Setup
imports and config

In [ ]:
import sys
sys.path.insert(0, 'src')
from collections import Counter
from config import load_config

cfg = load_config()
print('seed:', cfg['seed'], '| n_products_target:', cfg['data_source']['n_products_target'], '| n_questions:', cfg['questions']['n_total'])


## Stage 0 - Load data
input: public DM Search API  
output: `data/raw/*.json`  
description: loads DM products per search term; the only stage that touches the network, so it cannot reproduce the original fetch exactly since the catalog changes over time.

In [ ]:
import get_products
leaves = get_products.run(cfg, rerun=False)

## Stage 1 - Clean
input: `data/raw/*.json`  
output: `stage_1_products.json`  
description: dedupe, scrub real brand, parse descriptor/fill volume/price.

In [ ]:
import clean_products
products = clean_products.run(cfg, rerun=False)

## Stage 2 - Name pools (LLM)
input: config `names.counts`  
output: `stage_2_names.json`  
description: seed-deterministic invented names (brand/supplier/warehouse/region/buyer/team) via OSS-LLM.

In [ ]:
import gen_names
names = gen_names.run(cfg, rerun=False)

## Stage 3 - Build graph
input: `stage_1_products.json` + `stage_2_names.json`  
output: `stage_3_graph.json`  
description: wire nodes + edges uniform-random from seed.

In [ ]:
import build_graph
graph = build_graph.run(cfg, rerun=False)

## Stage 4 - Emit facts corpus
input: `stage_3_graph.json`  
output: `stage_4_corpus.jsonl` (+ Cache `stage_4_corpus_cache.json`)  
description: one facts doc per entity; LLM-weave per config `corpus.llm_weave`, cache per fact-hash.

In [ ]:
import emit_corpus

corpus = emit_corpus.run(cfg, rerun=False)

## Stage 5 - Generate questions
input: `stage_3_graph.json` + `stage_4_corpus.jsonl`  
output: `stage_5_questions_raw.json`  
description: multi-hop questions per category.

In [ ]:
import gen_questions

items = gen_questions.run(cfg, rerun=False)

## Stage 6 - Rewriting for TTS
input: `stage_5_questions_raw.json`  
output: `stage_6_questions_spoken.json` (field `spoken_question`)  
description: speakable form; runs **before** the closed-book gate so rewrite-induced answer leaks are still caught.

In [ ]:
import spoken_rewrite
spoken = spoken_rewrite.run(cfg, rerun=False)

## Stage 7 - Closed-Book Gate
input: `stage_6_questions_spoken.json`  
output: `stage_7_questions.json`  
description: ensemble of LLMs answers the spoken question without corpus; if a model guesses correctly → item removed.

In [ ]:
import qa_checks
final = qa_checks.run(cfg, gate=True, rerun=True)
print(f'kept {len(final)} items -> {cfg["_paths"]["questions"]}')

## Stage 7.5 - Pre-audio validation gate
input: `stage_7_questions.json` + `stage_4_corpus.jsonl`  
output: console report; **raises** on any content error  
description: deterministic, no-LLM, no-audio gate. Re-derives every answer from the corpus (argmax / sum / terminal hop) and asserts number fidelity, no verb drift, no answer leak, and the early/serial front-loading contract. Content bugs are the only failures that force a full dataset redo, so catch them here **before** spending time on TTS/recording. Audio-only issues (mis-synthesis, ASR) are self-gated per item by Stage 8/9 and never require a full rebuild.

In [ ]:
import importlib, validate
importlib.reload(validate)   # pick up edits without a kernel restart
errors, warns = validate.run(cfg)
for w in warns:
    print('WARN ', w)
for e in errors:
    print('ERROR', e)
assert not errors, f'{len(errors)} content errors — fix before recording audio (see above)'
print(f'VALIDATION PASSED — {len(warns)} warnings, safe for stage 8 (TTS) / stage 9 (recording)')

## Stage 8 - Synthetic TTS
input: `stage_7_questions.json`  
output: `audio_synthetic/*.wav`  
description: local OSS TTS (`tts.model_id`) per question; every WAV is ASR-verified via `tts.verify` and re-sampled on rejection, because this checkpoint silently truncates or babbles on long questions; `limit=` for test run.

In [ ]:
import oss_tts

audio_items = oss_tts.run(cfg, rerun=True)   # add limit=3 to test a few first

## Stage 9 - Human recording
input: `stage_7_questions.json`  
output: `audio_real/*.wav` (+ `speaker` per item in `stage_7_questions.json`)  
description: local recording UI at `recording.recorder.host`:`recording.recorder.port` (verbatim script, async fidelity gate); stop server with `Ctrl+C`. Two speakers share the set 50/50 stratified per category; the UI serves only the queue of `recording.currently_recording`, so run this cell once per speaker and flip that config key in between.

In [ ]:
import record_audio

record_audio.run(cfg, rerun=False)

## Stage 10 - Manifest export
input: `stage_7_questions.json` + Audio (`audio.source` in config)  
output: `manifest.json`  
description: question + chosen audio path → driver independent of source.

In [ ]:
import build_manifest
manifest = build_manifest.run()

## Stage 11 - Validation
input: `stage_7_questions.json` + `stage_4_corpus.jsonl`  
output: console report  
description: gold-vs-self scoring (should be 100%) + closed-book leak check (should be 0).

In [ ]:
import scoring
from common import read_json
final = read_json(cfg['_paths']['questions'])   # from disk → independent of cell order/kernel restart
tol = cfg['scoring']['number_tolerance']
self_ok = sum(
    scoring.score_item(
        list(it['gold_answer']) if it['answer_type'] == 'list' else it['gold_answer'], it, tol
    )
    for it in final
)
n_leak = sum(1 for i in final if any((i.get('closed_book_hits') or {}).values()))
print(f'gold-vs-self {self_ok}/{len(final)} (should be 100%) · leaks {n_leak} (should be 0) · {cfg["_paths"]["questions"]}')